# Public research notebook

This notebook is a cleaned public version of the original
research workflow.

## Execution model

- All filesystem paths are relative to the repository root.
- No external mounted filesystem is required.
- Stored cell outputs have been removed.
- Generated files are written below the local `results/`
  directory.
- The archival source notebook remains unchanged.


In [ ]:
# Portable repository configuration
#
# The notebook assumes that it is executed from the repository
# root or from a cloned copy of the repository.

from pathlib import Path

REPOSITORY_ROOT = Path.cwd().resolve()

# Move upward when the notebook is launched from a nested folder.
if REPOSITORY_ROOT.name in {
    "lorenz",
    "rossler",
    "duffing",
    "kuramoto",
    "stuart_landau",
    "coupled_map_lattice",
}:
    REPOSITORY_ROOT = REPOSITORY_ROOT.parents[1]

DATA_DIR = REPOSITORY_ROOT / "data"
RESULTS_DIR = REPOSITORY_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"

for directory in [
    DATA_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    CHECKPOINTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPOSITORY_ROOT}")
print(f"Results directory: {RESULTS_DIR}")


In [ ]:
# ============================================================
# DELTA WINDOW PROJECT — DUFFING PIPELINE INIT
# ============================================================


import os
import shutil
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

PROJECT_NAME = "delta_window_duffing_v2"
BASE_DIR = "data/raw"

PROJECT_DIR = f"{BASE_DIR}/{PROJECT_NAME}"
CSV_DIR = f"{PROJECT_DIR}/csv"
FIG_DIR = f"{PROJECT_DIR}/figures"
FINAL_FIG_DIR = f"{PROJECT_DIR}/final_figures"
CHECKPOINT_DIR = f"{PROJECT_DIR}/checkpoints"
LOG_DIR = f"{PROJECT_DIR}/logs"

for d in [PROJECT_DIR, CSV_DIR, FIG_DIR, FINAL_FIG_DIR, CHECKPOINT_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

def save_csv(df, name, folder=CSV_DIR):
    path = f"{folder}/{name}_{TIMESTAMP}.csv"
    df.to_csv(path, index=False)
    print(f"[CSV SAVED] {path}")
    return path

def save_checkpoint(df, name):
    path = f"{CHECKPOINT_DIR}/{name}_CHECKPOINT.csv"
    df.to_csv(path, index=False)
    print(f"[CHECKPOINT SAVED] {path}")
    return path

def save_figure(plt_obj, name, folder=FIG_DIR):
    path = f"{folder}/{name}_{TIMESTAMP}.png"
    plt_obj.savefig(path, dpi=300, bbox_inches="tight")
    print(f"[FIGURE SAVED] {path}")
    return path

def save_final_figure(plt_obj, name):
    path = f"{FINAL_FIG_DIR}/{name}_{TIMESTAMP}.png"
    plt_obj.savefig(path, dpi=300, bbox_inches="tight")
    print(f"[FINAL FIGURE SAVED] {path}")
    return path

with open(f"{LOG_DIR}/session_info_{TIMESTAMP}.json", "w") as f:
    json.dump(
        {"project_name": PROJECT_NAME, "timestamp": TIMESTAMP},
        f,
        indent=4
    )

print("\n===================================================")
print("DUFFING PIPELINE INITIALIZED")
print(PROJECT_DIR)
print("===================================================")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# parameters DUFFINGA
# =========================
# x'' + delta_d*x' - x + x^3 = gamma*cos(omega*t)

delta_d = 0.2   # tłumienie układu Duffinga
gamma = 0.3     # amplituda wymuszenia
omega = 1.2     # częstość wymuszenia

DT = 0.01
STEPS = 20000
RUNS = 10

delta_values = np.linspace(0.05, 2.0, 30)

# =========================
# SYMULACJA DUFFING + FILTR Δ
# =========================
def simulate_duffing(delta, damping, rng):
    # stan: x, v
    x = 0.1 + 0.01 * rng.normal()
    v = 0.0 + 0.01 * rng.normal()
    t = 0.0

    xs = []
    vs = []
    transitions = 0

    prev_sign = np.sign(x)
    filtered_steps = 0

    for _ in range(STEPS):
        # x' = v
        dx = v

        # v' = -delta_d*v + x - x^3 + gamma*cos(omega*t)
        dv = -delta_d * v + x - x**3 + gamma * np.cos(omega * t)

        new_x = x + dx * DT
        new_v = v + dv * DT

        # step length in the (x, v) state space
        jump = np.sqrt((new_x - x)**2 + (new_v - v)**2)

        # filtr Δ
        if jump > delta:
            scale = delta / jump
            new_x = x + (new_x - x) * scale * damping
            new_v = v + (new_v - v) * scale * damping
            filtered_steps += 1

        x, v = new_x, new_v
        t += DT

        xs.append(x)
        vs.append(v)

        sign = np.sign(x)
        if sign != 0 and prev_sign != 0 and sign != prev_sign:
            transitions += 1
        if sign != 0:
            prev_sign = sign

    return transitions, filtered_steps, np.array(xs), np.array(vs)

# =========================
# DWELL TIME
# =========================
def get_dwell_times(xs):
    xs = np.array(xs, dtype=float)
    signs = np.sign(xs)

    if len(signs) == 0:
        return np.array([])

    # zero padding
    if signs[0] == 0:
        signs[0] = 1

    for i in range(1, len(signs)):
        if signs[i] == 0:
            signs[i] = signs[i-1]

    dwell = []
    count = 1

    for i in range(1, len(signs)):
        if signs[i] == signs[i-1]:
            count += 1
        else:
            dwell.append(count)
            count = 1

    dwell.append(count)
    return np.array(dwell, dtype=float)

# =========================
# EXPLORATION
# =========================
def occupancy(xs, vs, bins=60):
    H, _, _ = np.histogram2d(xs, vs, bins=bins)
    return np.sum(H > 0)

In [ ]:
DAMPING = 0.5

mean_transitions = []
mean_dwell = []
mean_occupancy = []

std_transitions = []
std_dwell = []
std_occupancy = []

for delta in delta_values:
    t_runs = []
    d_runs = []
    o_runs = []

    for run in range(RUNS):
        rng = np.random.default_rng(run)

        t, filtered_steps, xs, vs = simulate_duffing(delta, DAMPING, rng)

        dwell = get_dwell_times(xs)
        mean_d = np.mean(dwell) * DT if len(dwell) > 0 else np.nan
        occ = occupancy(xs, vs)

        t_runs.append(t)
        d_runs.append(mean_d)
        o_runs.append(occ)

    mean_transitions.append(np.mean(t_runs))
    mean_dwell.append(np.mean(d_runs))
    mean_occupancy.append(np.mean(o_runs))

    std_transitions.append(np.std(t_runs))
    std_dwell.append(np.std(d_runs))
    std_occupancy.append(np.std(o_runs))

In [ ]:
plt.figure(figsize=(8,5))
plt.errorbar(delta_values, mean_transitions, yerr=std_transitions, fmt='o-', capsize=4)
plt.title("Transitions vs Delta (Duffing)")
plt.xlabel("Delta")
plt.ylabel("Mean transitions")
plt.grid()
plt.show()

plt.figure(figsize=(8,5))
plt.errorbar(delta_values, mean_dwell, yerr=std_dwell, fmt='o-', capsize=4)
plt.title("Dwell vs Delta (Duffing)")
plt.xlabel("Delta")
plt.ylabel("Mean dwell time")
plt.grid()
plt.show()

plt.figure(figsize=(8,5))
plt.errorbar(delta_values, mean_occupancy, yerr=std_occupancy, fmt='o-', capsize=4)
plt.title("Exploration vs Delta (Duffing)")
plt.xlabel("Delta")
plt.ylabel("Mean occupancy")
plt.grid()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# parameters DUFFINGA
# =========================
# x'' + delta_d*x' - x + x^3 = gamma*cos(omega*t)

delta_d = 0.2
gamma = 0.5
omega = 1.0

DT = 0.01
STEPS = 20000
RUNS = 10

delta_values = np.linspace(0.05, 2.0, 30)

# =========================
# SYMULACJA DUFFING + FILTR Δ
# =========================
def simulate_duffing(delta, damping, rng):
    x = 0.1 + 0.01 * rng.normal()
    v = 0.0 + 0.01 * rng.normal()
    t = 0.0

    xs = []
    vs = []
    transitions = 0
    filtered_steps = 0

    prev_sign = np.sign(x)
    if prev_sign == 0:
        prev_sign = 1

    for _ in range(STEPS):
        dx = v
        dv = -delta_d * v + x - x**3 + gamma * np.cos(omega * t)

        new_x = x + dx * DT
        new_v = v + dv * DT

        jump = np.sqrt((new_x - x)**2 + (new_v - v)**2)

        if jump > delta:
            scale = delta / jump
            new_x = x + (new_x - x) * scale * damping
            new_v = v + (new_v - v) * scale * damping
            filtered_steps += 1

        x, v = new_x, new_v
        t += DT

        xs.append(x)
        vs.append(v)

        sign = np.sign(x)
        if sign == 0:
            sign = prev_sign

        if sign != prev_sign:
            transitions += 1

        prev_sign = sign

    return transitions, filtered_steps, np.array(xs, dtype=float), np.array(vs, dtype=float)

# =========================
# DWELL TIME
# =========================
def get_dwell_times(xs):
    xs = np.array(xs, dtype=float)
    signs = np.sign(xs)

    if len(signs) == 0:
        return np.array([])

    if signs[0] == 0:
        signs[0] = 1

    for i in range(1, len(signs)):
        if signs[i] == 0:
            signs[i] = signs[i - 1]

    dwell = []
    count = 1

    for i in range(1, len(signs)):
        if signs[i] == signs[i - 1]:
            count += 1
        else:
            dwell.append(count)
            count = 1

    dwell.append(count)
    return np.array(dwell, dtype=float)

# =========================
# EXPLORATION
# =========================
def occupancy(xs, vs, bins=60):
    H, _, _ = np.histogram2d(xs, vs, bins=bins)
    return np.sum(H > 0)

# =========================
# TEST PODSTAWOWY
# =========================
DAMPING = 0.5

mean_transitions = []
mean_dwell = []
mean_occupancy = []

std_transitions = []
std_dwell = []
std_occupancy = []

mean_filtered = []
std_filtered = []

for delta in delta_values:
    t_runs = []
    d_runs = []
    o_runs = []
    f_runs = []

    for run in range(RUNS):
        rng = np.random.default_rng(run)

        t, filtered_steps, xs, vs = simulate_duffing(delta, DAMPING, rng)

        dwell = get_dwell_times(xs)
        mean_d = np.mean(dwell) * DT if len(dwell) > 0 else np.nan
        occ = occupancy(xs, vs)

        t_runs.append(t)
        d_runs.append(mean_d)
        o_runs.append(occ)
        f_runs.append(filtered_steps)

    mean_transitions.append(np.mean(t_runs))
    mean_dwell.append(np.mean(d_runs))
    mean_occupancy.append(np.mean(o_runs))
    mean_filtered.append(np.mean(f_runs))

    std_transitions.append(np.std(t_runs))
    std_dwell.append(np.std(d_runs))
    std_occupancy.append(np.std(o_runs))
    std_filtered.append(np.std(f_runs))

# =========================
# plots
# =========================
plt.figure(figsize=(8,5))
plt.errorbar(delta_values, mean_transitions, yerr=std_transitions, fmt='o-', capsize=4)
plt.title("Transitions vs Delta (Duffing)")
plt.xlabel("Delta")
plt.ylabel("Mean transitions")
plt.grid()
plt.show()

plt.figure(figsize=(8,5))
plt.errorbar(delta_values, mean_dwell, yerr=std_dwell, fmt='o-', capsize=4)
plt.title("Dwell vs Delta (Duffing)")
plt.xlabel("Delta")
plt.ylabel("Mean dwell time")
plt.grid()
plt.show()

plt.figure(figsize=(8,5))
plt.errorbar(delta_values, mean_occupancy, yerr=std_occupancy, fmt='o-', capsize=4)
plt.title("Exploration vs Delta (Duffing)")
plt.xlabel("Delta")
plt.ylabel("Mean occupancy")
plt.grid()
plt.show()

plt.figure(figsize=(8,5))
plt.errorbar(delta_values, mean_filtered, yerr=std_filtered, fmt='o-', capsize=4)
plt.title("Filtered steps vs Delta (Duffing)")
plt.xlabel("Delta")
plt.ylabel("Mean filtered steps")
plt.grid()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# parameters DUFFINGA
# =========================
delta_d = 0.2
gamma = 0.5
omega = 1.0

DT = 0.01
STEPS = 20000
RUNS = 10

# Extend the scan to much smaller Delta values
delta_values = np.linspace(0.001, 0.1, 30)

# =========================
# SYMULACJA DUFFING + FILTR Δ
# =========================
def simulate_duffing(delta, damping, rng):
    x = 0.1 + 0.01 * rng.normal()
    v = 0.0 + 0.01 * rng.normal()
    t = 0.0

    xs = []
    vs = []
    transitions = 0
    filtered_steps = 0

    prev_sign = np.sign(x)
    if prev_sign == 0:
        prev_sign = 1

    for _ in range(STEPS):
        dx = v
        dv = -delta_d * v + x - x**3 + gamma * np.cos(omega * t)

        new_x = x + dx * DT
        new_v = v + dv * DT

        jump = np.sqrt((new_x - x)**2 + (new_v - v)**2)

        if jump > delta:
            scale = delta / jump
            new_x = x + (new_x - x) * scale * damping
            new_v = v + (new_v - v) * scale * damping
            filtered_steps += 1

        x, v = new_x, new_v
        t += DT

        xs.append(x)
        vs.append(v)

        sign = np.sign(x)
        if sign == 0:
            sign = prev_sign

        if sign != prev_sign:
            transitions += 1

        prev_sign = sign

    return transitions, filtered_steps, np.array(xs, dtype=float), np.array(vs, dtype=float)

# =========================
# DWELL TIME
# =========================
def get_dwell_times(xs):
    xs = np.array(xs, dtype=float)
    signs = np.sign(xs)

    if len(signs) == 0:
        return np.array([])

    if signs[0] == 0:
        signs[0] = 1

    for i in range(1, len(signs)):
        if signs[i] == 0:
            signs[i] = signs[i - 1]

    dwell = []
    count = 1

    for i in range(1, len(signs)):
        if signs[i] == signs[i - 1]:
            count += 1
        else:
            dwell.append(count)
            count = 1

    dwell.append(count)
    return np.array(dwell, dtype=float)

# =========================
# EXPLORATION
# =========================
def occupancy(xs, vs, bins=60):
    H, _, _ = np.histogram2d(xs, vs, bins=bins)
    return np.sum(H > 0)

# =========================
# SMALL-DELTA TEST
# =========================
DAMPING = 0.5

mean_transitions = []
mean_dwell = []
mean_occupancy = []
mean_filtered = []

std_transitions = []
std_dwell = []
std_occupancy = []
std_filtered = []

for delta in delta_values:
    t_runs = []
    d_runs = []
    o_runs = []
    f_runs = []

    for run in range(RUNS):
        rng = np.random.default_rng(run)

        t, filtered_steps, xs, vs = simulate_duffing(delta, DAMPING, rng)

        dwell = get_dwell_times(xs)
        mean_d = np.mean(dwell) * DT if len(dwell) > 0 else np.nan
        occ = occupancy(xs, vs)

        t_runs.append(t)
        d_runs.append(mean_d)
        o_runs.append(occ)
        f_runs.append(filtered_steps)

    mean_transitions.append(np.mean(t_runs))
    mean_dwell.append(np.mean(d_runs))
    mean_occupancy.append(np.mean(o_runs))
    mean_filtered.append(np.mean(f_runs))

    std_transitions.append(np.std(t_runs))
    std_dwell.append(np.std(d_runs))
    std_occupancy.append(np.std(o_runs))
    std_filtered.append(np.std(f_runs))

# =========================
# plots
# =========================
plt.figure(figsize=(8,5))
plt.errorbar(delta_values, mean_filtered, yerr=std_filtered, fmt='o-', capsize=4)
plt.title("Filtered steps vs Delta (Duffing, small Delta)")
plt.xlabel("Delta")
plt.ylabel("Mean filtered steps")
plt.grid()
plt.show()

plt.figure(figsize=(8,5))
plt.errorbar(delta_values, mean_transitions, yerr=std_transitions, fmt='o-', capsize=4)
plt.title("Transitions vs Delta (Duffing, small Delta)")
plt.xlabel("Delta")
plt.ylabel("Mean transitions")
plt.grid()
plt.show()

plt.figure(figsize=(8,5))
plt.errorbar(delta_values, mean_dwell, yerr=std_dwell, fmt='o-', capsize=4)
plt.title("Dwell vs Delta (Duffing, small Delta)")
plt.xlabel("Delta")
plt.ylabel("Mean dwell time")
plt.grid()
plt.show()

plt.figure(figsize=(8,5))
plt.errorbar(delta_values, mean_occupancy, yerr=std_occupancy, fmt='o-', capsize=4)
plt.title("Exploration vs Delta (Duffing, small Delta)")
plt.xlabel("Delta")
plt.ylabel("Mean occupancy")
plt.grid()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# PARAMETERS (same as above)
# =========================
delta_d = 0.2
gamma = 0.5
omega = 1.0

DT = 0.01
STEPS = 20000
RUNS = 10

delta_values = np.linspace(0.001, 0.1, 30)
DAMPING = 0.5

# =========================
# SYMULACJA DUFFING + Δ
# =========================
def simulate_duffing(delta, damping, rng):
    x = 0.1 + 0.01 * rng.normal()
    v = 0.0 + 0.01 * rng.normal()
    t = 0.0

    xs = []
    vs = []
    transitions = 0
    filtered_steps = 0

    prev_sign = np.sign(x)
    if prev_sign == 0:
        prev_sign = 1

    for _ in range(STEPS):
        dx = v
        dv = -delta_d * v + x - x**3 + gamma * np.cos(omega * t)

        new_x = x + dx * DT
        new_v = v + dv * DT

        jump = np.sqrt((new_x - x)**2 + (new_v - v)**2)

        if jump > delta:
            scale = delta / jump
            new_x = x + (new_x - x) * scale * damping
            new_v = v + (new_v - v) * scale * damping
            filtered_steps += 1

        x, v = new_x, new_v
        t += DT

        xs.append(x)
        vs.append(v)

        sign = np.sign(x)
        if sign == 0:
            sign = prev_sign

        if sign != prev_sign:
            transitions += 1

        prev_sign = sign

    return transitions, filtered_steps, np.array(xs, dtype=float), np.array(vs, dtype=float)

# =========================
# DWELL TIME
# =========================
def get_dwell_times(xs):
    xs = np.array(xs, dtype=float)
    signs = np.sign(xs)

    if len(signs) == 0:
        return np.array([])

    if signs[0] == 0:
        signs[0] = 1

    for i in range(1, len(signs)):
        if signs[i] == 0:
            signs[i] = signs[i - 1]

    dwell = []
    count = 1

    for i in range(1, len(signs)):
        if signs[i] == signs[i - 1]:
            count += 1
        else:
            dwell.append(count)
            count = 1

    dwell.append(count)
    return np.array(dwell, dtype=float)

# =========================
# EXPLORATION
# =========================
def occupancy(xs, vs, bins=60):
    H, _, _ = np.histogram2d(xs, vs, bins=bins)
    return np.sum(H > 0)

# =========================
# MEAN STEP LENGTH (KEY QUANTITY)
# =========================
def mean_step_size(xs, vs):
    dx = np.diff(xs)
    dv = np.diff(vs)
    steps = np.sqrt(dx**2 + dv**2)
    return np.mean(steps)

# =========================
# MAIN TEST AND NORMALIZATION
# =========================
mean_transitions = []
mean_dwell = []
mean_occupancy = []
mean_filtered = []
mean_step = []

for delta in delta_values:
    t_runs = []
    d_runs = []
    o_runs = []
    f_runs = []
    s_runs = []

    for run in range(RUNS):
        rng = np.random.default_rng(run)

        t, filtered_steps, xs, vs = simulate_duffing(delta, DAMPING, rng)

        dwell = get_dwell_times(xs)
        mean_d = np.mean(dwell) * DT if len(dwell) > 0 else np.nan
        occ = occupancy(xs, vs)
        step = mean_step_size(xs, vs)

        t_runs.append(t)
        d_runs.append(mean_d)
        o_runs.append(occ)
        f_runs.append(filtered_steps)
        s_runs.append(step)

    mean_transitions.append(np.mean(t_runs))
    mean_dwell.append(np.mean(d_runs))
    mean_occupancy.append(np.mean(o_runs))
    mean_filtered.append(np.mean(f_runs))
    mean_step.append(np.mean(s_runs))

# =========================
# NORMALIZACJA
# =========================
delta_norm = np.array(delta_values) / np.array(mean_step)

# =========================
# plots (ZNORMALIZOWANE)
# =========================
plt.figure(figsize=(8,5))
plt.plot(delta_norm, mean_transitions, 'o-')
plt.title("Transitions vs normalized Delta (Duffing)")
plt.xlabel("Delta / mean_step")
plt.grid()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(delta_norm, mean_dwell, 'o-')
plt.title("Dwell vs normalized Delta (Duffing)")
plt.xlabel("Delta / mean_step")
plt.grid()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(delta_norm, mean_occupancy, 'o-')
plt.title("Exploration vs normalized Delta (Duffing)")
plt.xlabel("Delta / mean_step")
plt.grid()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(delta_norm, mean_filtered, 'o-')
plt.title("Filtered steps vs normalized Delta (Duffing)")
plt.xlabel("Delta / mean_step")
plt.grid()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# INSERT THE ARRAYS FROM THE THREE NOTEBOOKS HERE
# ==========================================

# --- LORENZ ---
# wklej z notebooka Lorenza
lorenz_delta_norm = np.array([
    # np. 0.8, 1.0, 1.2, ...
])

lorenz_exploration = np.array([
    # mean_occupancy, mean_range_z, or another metric
])

lorenz_filtered = np.array([
    # mean_filtered_steps
])

# --- ROSSLER ---
rossler_delta_norm = np.array([
    # ...
])

rossler_exploration = np.array([
    # ...
])

rossler_filtered = np.array([
    # ...
])

# --- DUFFING ---
duffing_delta_norm = np.array([
    # ...
])

duffing_exploration = np.array([
    # ...
])

duffing_filtered = np.array([
    # ...
])

# ==========================================
# FUNCTION THAT NORMALIZES Y TO [0, 1]
# ==========================================
def normalize_curve(y):
    y = np.array(y, dtype=float)
    ymin = np.nanmin(y)
    ymax = np.nanmax(y)
    if ymax - ymin == 0:
        return np.zeros_like(y)
    return (y - ymin) / (ymax - ymin)

# ==========================================
# NORMALIZACJA KRZYWYCH
# ==========================================
lorenz_exploration_n = normalize_curve(lorenz_exploration)
rossler_exploration_n = normalize_curve(rossler_exploration)
duffing_exploration_n = normalize_curve(duffing_exploration)

lorenz_filtered_n = normalize_curve(lorenz_filtered)
rossler_filtered_n = normalize_curve(rossler_filtered)
duffing_filtered_n = normalize_curve(duffing_filtered)

# ==========================================
# plot 1: EXPLORATION OVERLAY
# ==========================================
plt.figure(figsize=(9,6))

plt.plot(lorenz_delta_norm, lorenz_exploration_n, 'o-', label='Lorenz')
plt.plot(rossler_delta_norm, rossler_exploration_n, 'o-', label='Rossler')
plt.plot(duffing_delta_norm, duffing_exploration_n, 'o-', label='Duffing')

plt.xlabel("Delta / mean_step")
plt.ylabel("Normalized exploration")
plt.title("Overlay: normalized exploration vs normalized Delta")
plt.grid(True)
plt.legend()
plt.show()

# ==========================================
# plot 2: FILTERED STEPS OVERLAY
# ==========================================
plt.figure(figsize=(9,6))

plt.plot(lorenz_delta_norm, lorenz_filtered_n, 'o-', label='Lorenz')
plt.plot(rossler_delta_norm, rossler_filtered_n, 'o-', label='Rossler')
plt.plot(duffing_delta_norm, duffing_filtered_n, 'o-', label='Duffing')

plt.xlabel("Delta / mean_step")
plt.ylabel("Normalized filtered steps")
plt.title("Overlay: normalized filtering vs normalized Delta")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
np.savez("duffing_data.npz",
         delta_norm=np.array(delta_norm, dtype=float),
         exploration=np.array(mean_occupancy, dtype=float),
         filtered=np.array(mean_filtered, dtype=float))

print("saved duffing_data.npz")

In [ ]:
# ============================================================
# FINAL UNIVERSAL SAVE BLOCK — DUFFING
# ============================================================

import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

FINAL_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("\n===================================================")
print("STARTING FINAL AUTO SAVE — DUFFING")
print("===================================================")

# ============================================================
# SAVE ALL DATAFRAMES
# ============================================================

saved_dfs = []

for var_name, var_value in list(globals().items()):

    if isinstance(var_value, pd.DataFrame):

        try:
            save_path = f"{CSV_DIR}/{var_name}_{FINAL_TIMESTAMP}.csv"
            var_value.to_csv(save_path, index=False)

            saved_dfs.append({
                "name": var_name,
                "rows": len(var_value),
                "columns": len(var_value.columns),
                "path": save_path
            })

            print(f"[DATAFRAME SAVED] {var_name}")
            print(save_path)

        except Exception as e:
            print(f"[ERROR SAVING {var_name}] {e}")

index_df = pd.DataFrame(saved_dfs)
index_path = f"{CSV_DIR}/RECOVERED_DATAFRAME_INDEX_{FINAL_TIMESTAMP}.csv"
index_df.to_csv(index_path, index=False)

print("\n[DATAFRAME INDEX SAVED]")
print(index_path)

# ============================================================
# SAVE IMPORTANT DUFFING LISTS / ARRAYS
# ============================================================

array_names = [
    "delta_values",
    "mean_transitions",
    "std_transitions",
    "mean_dwell",
    "std_dwell",
    "mean_occupancy",
    "std_occupancy",
    "mean_filtered",
    "std_filtered",
    "delta_norm",
]

saved_arrays = {}

for name in array_names:
    if name in globals():
        try:
            saved_arrays[name] = np.array(globals()[name], dtype=float)
            print(f"[ARRAY FOUND] {name}, shape={saved_arrays[name].shape}")
        except Exception as e:
            print(f"[ARRAY ERROR] {name}: {e}")

if len(saved_arrays) > 0:
    npz_path = f"{CSV_DIR}/duffing_recovered_arrays_{FINAL_TIMESTAMP}.npz"
    np.savez(npz_path, **saved_arrays)
    print("\n[ARRAY NPZ SAVED]")
    print(npz_path)

# ============================================================
# CREATE DUFFING SUMMARY TABLE IF ARRAYS EXIST
# ============================================================

try:
    if "delta_values" in globals():

        n = len(delta_values)

        summary_dict = {"delta": np.array(delta_values, dtype=float)}

        possible_cols = {
            "mean_transitions": "mean_transitions",
            "std_transitions": "std_transitions",
            "mean_dwell": "mean_dwell",
            "std_dwell": "std_dwell",
            "mean_occupancy": "mean_occupancy",
            "std_occupancy": "std_occupancy",
            "mean_filtered": "mean_filtered",
            "std_filtered": "std_filtered",
        }

        for var, col in possible_cols.items():
            if var in globals() and len(globals()[var]) == n:
                summary_dict[col] = np.array(globals()[var], dtype=float)

        duffing_summary_df = pd.DataFrame(summary_dict)

        summary_path = f"{CSV_DIR}/duffing_summary_recovered_{FINAL_TIMESTAMP}.csv"
        duffing_summary_df.to_csv(summary_path, index=False)

        print("\n[DUFFING SUMMARY SAVED]")
        print(summary_path)

except Exception as e:
    print("[DUFFING SUMMARY ERROR]")
    print(e)

# ============================================================
# SAFE NPZ EXPORT — ONLY IF VARIABLES EXIST
# ============================================================

try:
    if all(x in globals() for x in ["delta_norm", "mean_occupancy", "mean_filtered"]):

        npz_path = f"{CSV_DIR}/duffing_data_{FINAL_TIMESTAMP}.npz"

        np.savez(
            npz_path,
            delta_norm=np.array(delta_norm, dtype=float),
            exploration=np.array(mean_occupancy, dtype=float),
            filtered=np.array(mean_filtered, dtype=float)
        )

        print("\n[DUFFING DATA NPZ SAVED]")
        print(npz_path)

    else:
        print("\n[SKIPPED duffing_data.npz]")
        print("Missing one of: delta_norm, mean_occupancy, mean_filtered")

except Exception as e:
    print("[DUFFING NPZ ERROR]")
    print(e)

# ============================================================
# SAVE ALL OPEN FIGURES
# ============================================================

fig_nums = plt.get_fignums()
saved_figs = []

print(f"\nOpen figures found: {len(fig_nums)}")

for i, fig_num in enumerate(fig_nums, start=1):

    try:
        fig = plt.figure(fig_num)
        fig_path = f"{FIG_DIR}/duffing_recovered_figure_{i}_{FINAL_TIMESTAMP}.png"
        fig.savefig(fig_path, dpi=300, bbox_inches="tight")

        saved_figs.append(fig_path)

        print(f"[FIGURE SAVED]")
        print(fig_path)

    except Exception as e:
        print(f"[ERROR SAVING FIGURE {i}] {e}")

# ============================================================
# ZIP BACKUP
# ============================================================

try:
    zip_path = f"{BASE_DIR}/{PROJECT_NAME}_FULL_BACKUP_{FINAL_TIMESTAMP}"

    shutil.make_archive(
        zip_path,
        "zip",
        PROJECT_DIR
    )

    print("\n[FULL ZIP BACKUP CREATED]")
    print(f"{zip_path}.zip")

except Exception as e:
    print("[ZIP BACKUP ERROR]")
    print(e)

print("\n===================================================")
print("FINAL SAVE COMPLETED — DUFFING")
print(f"DataFrames saved: {len(saved_dfs)}")
print(f"Figures saved: {len(saved_figs)}")
print("===================================================")